# 08 · Split-Half Reliability and Precision

**Paper**: Rahnev et al., *Nature Communications 2025*  
**MATLAB scripts**: `ana_splitHalf.m`, `ana_precision.m`

## Overview

### 1. Split-Half Reliability
How consistently does each measure reproduce itself when computed on two independent halves 
of the data (odd vs. even trials)?  High reliability = the measure is stable, not noisy.

**Spearman-Brown correction**: $r_{\text{full}} = 2r_{\text{half}} / (1 + r_{\text{half}})$

### 2. Precision
How sensitive is each measure to a known, artificially induced change in metacognitive accuracy?  
We corrupt a fraction of trials (2%, 4%, 6%) and measure the drop in each metacognitive measure.

> **Note**: For precision, MLE-based measures (meta-d', M-Ratio, M-Diff) use non-MLE approximations 
> for speed. This matches the spirit of the MATLAB analysis — see `ana_precision.m`.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import sys, os, warnings
warnings.filterwarnings('ignore')

REPO = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO, 'src'))
sys.path.insert(0, os.path.join(REPO, 'notebooks'))
OUT = os.path.join(REPO, 'notebooks', 'precomputed')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from analysis_core import (
    MEASURE_NAMES, N_MEASURES,
    preprocess_haddara, preprocess_maniscalco, preprocess_shekhar,
)
from metasignal.stdpy.core import compute_sdt_resp, trials_to_counts
from metasignal.stdpy.type2 import (
    sdt_expect_conf, compute_type2_auc, compute_gamma, compute_phi, compute_delta_conf
)

def r2z(r): return np.arctanh(np.clip(r, -0.9999, 0.9999))
def z2r(z): return np.tanh(z)

print('Imports OK')


## Fast measure computation (no MLE)

For precision analysis we need to compute measures ~200 times per run.  
MLE meta-d' takes ~3s/call; non-MLE measures take ~3ms/call.  
This function computes AUC2, Gamma, Phi, ΔConf and their normalized variants quickly.


In [ ]:
def fast_measures(stim, resp, conf, n_ratings):
    """Compute non-MLE measures only (fast). Returns 20-element array with NaN for MLE measures."""
    stim = np.asarray(stim, dtype=float)
    resp = np.asarray(resp, dtype=float)
    conf = np.asarray(conf, dtype=float)
    valid = ~np.isnan(stim) & ~np.isnan(resp) & ~np.isnan(conf)
    stim, resp, conf = stim[valid], resp[valid], conf[valid]
    if len(stim) == 0: return np.full(20, np.nan)

    sb = (stim == np.max(stim)).astype(int)
    rb = (resp == np.max(resp)).astype(int)
    dp, c, _ = compute_sdt_resp(sb, rb)
    mc = np.mean(conf)

    if np.array_equal(sb, rb) or dp == 0 or len(np.unique(conf)) == 1:
        return np.full(20, np.nan)

    n1, n2 = np.array(trials_to_counts(sb, rb, conf.astype(int), n_ratings))
    se = sdt_expect_conf(n1, n2)
    ne1, ne2 = np.array(se['nR_S1_exp']), np.array(se['nR_S2_exp'])
    a = compute_type2_auc(n1, n2); ae = compute_type2_auc(ne1, ne2)
    g = compute_gamma(n1, n2); ge = compute_gamma(ne1, ne2)
    ph = compute_phi(n1, n2); pe = compute_phi(ne1, ne2)
    dc = compute_delta_conf(n1, n2)

    return np.array([
        np.nan, a, g, ph, dc['delta_conf'],              # meta-d' skipped (slow)
        np.nan, a/ae if ae else np.nan,                  # M-Ratio skipped
        g/ge if ge else np.nan, ph/pe if pe else np.nan,
        dc['delta_conf_ratio'],
        np.nan, a-ae, g-ge, ph-pe, dc['delta_conf_diff'],  # M-Diff skipped
        np.nan, np.nan,                                   # meta-noise/uncertainty skipped
        float(dp), float(c), float(mc)
    ])

# Test
ha_subs = preprocess_haddara()
s0 = ha_subs[0]
r = fast_measures(s0['stim'], s0['resp'], s0['conf'], s0['n_ratings'])
print('Fast measures OK:', r[1:6].round(3))  # AUC2, Gamma, Phi, DeltaConf


## Part 1: Split-Half Reliability

Load precomputed odd/even split arrays.


In [ ]:
# Load split-half arrays (precomputed with MLE in 02_compute_measures.ipynb)
ha_split = np.load(os.path.join(OUT, 'haddara_mle.npz'))['split']     # (70, 2, 20)
ma_split = np.load(os.path.join(OUT, 'maniscalco_mle.npz'))['split']  # (22, 2, 20)

print(f'Haddara split:    {ha_split.shape}  (subjects, [odd, even], measures)')
print(f'Maniscalco split: {ma_split.shape}')

# Compute Shekhar split-half from raw data (fast)
print('Computing Shekhar split-half...')
sh_subs = preprocess_shekhar()
sh_split = np.full((len(sh_subs), 2, N_MEASURES), np.nan)
for i, s in enumerate(sh_subs):
    sh_split[i, 0] = fast_measures(s['stim'][0::2], s['resp'][0::2], s['conf'][0::2], s['n_ratings'])
    sh_split[i, 1] = fast_measures(s['stim'][1::2], s['resp'][1::2], s['conf'][1::2], s['n_ratings'])
print(f'Shekhar split:    {sh_split.shape}')


In [ ]:
def split_half_r(split_arr):
    """Correlate odd vs even trials across subjects for each measure."""
    rs = []
    for m in range(N_MEASURES):
        x, y = split_arr[:, 0, m], split_arr[:, 1, m]
        valid = ~np.isnan(x) & ~np.isnan(y)
        if valid.sum() < 3:
            rs.append(np.nan)
        else:
            r, _ = stats.pearsonr(x[valid], y[valid])
            rs.append(r)
    return np.array(rs)

def spearman_brown(r):
    """Correct half-test r to full-test r."""
    return np.where(np.isnan(r), np.nan, 2 * r / (1 + r))

r_ha = spearman_brown(split_half_r(ha_split))
r_ma = spearman_brown(split_half_r(ma_split))
r_sh = spearman_brown(split_half_r(sh_split))

# Average (Fisher z)
r_avg = np.array([z2r(np.nanmean(r2z(np.array([r_ha[m], r_ma[m], r_sh[m]]))))
                  for m in range(N_MEASURES)])

print('Split-half reliability (Spearman-Brown corrected):')
print(f'{"Measure":20s} {"Haddara":>10} {"Maniscalco":>12} {"Shekhar":>10} {"Average":>10}')
print('-'*66)
for m, name in enumerate(MEASURE_NAMES):
    print(f'{name:20s} {r_ha[m]:10.3f} {r_ma[m]:12.3f} {r_sh[m]:10.3f} {r_avg[m]:10.3f}')


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
width = 0.25
x = np.arange(N_MEASURES)
colors_ds = ['#e74c3c', '#3498db', '#2ecc71']
labels_ds = ['Haddara (n=70)', 'Maniscalco (n=22)', 'Shekhar (n=20)']

for di, (r_arr, lbl) in enumerate(zip([r_ha, r_ma, r_sh], labels_ds)):
    ax.bar(x + di*width, np.nan_to_num(r_arr), width, color=colors_ds[di], label=lbl, alpha=0.8)

ax.plot(x + width, np.where(np.isnan(r_avg), 0, r_avg), 'ko-', ms=5, lw=2, label='Average')
ax.axhline(0.7, color='gray', lw=1, ls='--', alpha=0.6, label='r=0.7 threshold')
ax.axhline(0, color='k', lw=0.5)
ax.set_xticks(x + width)
ax.set_xticklabels(MEASURE_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Spearman-Brown corrected r', fontsize=11)
ax.set_ylim([-0.1, 1.1])
ax.set_title('Split-Half Reliability', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(REPO, 'notebooks', 'split_half_reliability.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f'Mean r (17 measures): {z2r(np.nanmean(r2z(r_avg[:17]))):0.3f}')


## Part 2: Precision Analysis

We corrupt confidence ratings to reduce metacognitive accuracy by a known amount,
then measure how much each measure drops (in SD units).

**Protocol**: on correct trials with conf > 1, lower conf by 1;  
on error trials with conf < max, raise conf by 1;  
applied to 2%, 4%, 6% of trials.


In [ ]:
def corrupt_confidence(stim, resp, conf, n_ratings, prop_alter):
    """Corrupt confidence on prop_alter fraction of trials to reduce metacognitive accuracy."""
    n_alter = int(round(len(conf) * prop_alter))
    conf_new = conf.copy().astype(float)
    altered = 0
    for i in range(len(conf)):
        if altered >= n_alter: break
        if stim[i] == resp[i] and conf_new[i] > 1:
            conf_new[i] -= 1; altered += 1
        elif stim[i] != resp[i] and conf_new[i] < n_ratings:
            conf_new[i] += 1; altered += 1
    return conf_new

# Compute precision for Haddara
props = [0.02, 0.04, 0.06]
ha_orig = np.load(os.path.join(OUT, 'haddara_mle.npz'))['raw']  # (70, 20) — MLE measures

print(f'Computing precision for {len(ha_subs)} Haddara subjects × 3 corruption levels...')
ha_corrupted = np.full((len(ha_subs), len(props), N_MEASURES), np.nan)
for i, s in enumerate(ha_subs):
    for pi, prop in enumerate(props):
        conf_c = corrupt_confidence(s['stim'], s['resp'], s['conf'], s['n_ratings'], prop)
        ha_corrupted[i, pi] = fast_measures(s['stim'], s['resp'], conf_c, s['n_ratings'])
print('Done.')


In [ ]:
# Precision = (original - corrupted) / SD(original)
OUTLIER_CUT = 4.5

# Use fast (non-MLE) original measures for the non-MLE denominator
ha_fast_orig = np.full((len(ha_subs), N_MEASURES), np.nan)
for i, s in enumerate(ha_subs):
    ha_fast_orig[i] = fast_measures(s['stim'], s['resp'], s['conf'], s['n_ratings'])

ha_fast_orig[np.abs(ha_fast_orig) > OUTLIER_CUT] = np.nan

precision = np.full((len(props), N_MEASURES), np.nan)
for pi in range(len(props)):
    corrupted = ha_corrupted[:, pi, :].copy()
    corrupted[np.abs(corrupted) > OUTLIER_CUT] = np.nan
    diff = ha_fast_orig - corrupted
    sd_orig = np.nanstd(ha_fast_orig, axis=0, ddof=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        precision[pi] = np.nanmean(diff, axis=0) / sd_orig

# Average precision across all levels
avg_prec = np.nanmean(precision, axis=0)
avg16 = np.nanmean(avg_prec[np.array([1,2,3,4,6,7,8,9,11,12,13,14,17,18,19])])  # non-MLE measures
norm_prec = avg_prec / (avg16 if avg16 > 0 else 1)

print('Average precision by measure (SD units, averaged over 2/4/6%):')
non_mle = ['AUC2','Gamma','Phi','DeltaConf','AUC2-Ratio','Gamma-Ratio','Phi-Ratio',
           'DeltaConf-Ratio','AUC2-Diff','Gamma-Diff','Phi-Diff','DeltaConf-Diff',
           "d'", 'Criterion', 'Confidence']
for m, name in enumerate(MEASURE_NAMES):
    if name in non_mle:
        print(f'  {name:20s}: {avg_prec[m]:.3f}  (normalized: {norm_prec[m]:.3f})')


In [ ]:
# Visualize precision
fig, axes = plt.subplots(3, 5, figsize=(16, 9))
axes = axes.flatten()
plot_measures = [1,2,3,4,6,7,8,9,11,12,13,14,17,18,19]  # indices of non-MLE measures
prop_labels = ['2%', '4%', '6%']
colors_p = ['#9b59b6', '#e67e22', '#27ae60']

for pi_plot, mi in enumerate(plot_measures):
    ax = axes[pi_plot]
    vals = precision[:, mi]
    ax.bar(range(1, 4), vals, color=colors_p, alpha=0.8)
    ax.set_title(MEASURE_NAMES[mi], fontsize=9)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(prop_labels, fontsize=8)
    ax.set_ylim([-0.1, 1.8])
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(labelsize=7)

fig.suptitle('Precision: sensitivity to corrupted confidence (Haddara, non-MLE measures)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig(os.path.join(REPO, 'notebooks', 'precision.png'), dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Normalized precision bar chart
fig, ax = plt.subplots(figsize=(12, 4))
meas_names_plot = [MEASURE_NAMES[mi] for mi in plot_measures]
norm_vals = [norm_prec[mi] for mi in plot_measures]
ax.bar(range(len(plot_measures)), norm_vals,
       color=['#e74c3c' if v > 1.0 else '#3498db' for v in norm_vals], alpha=0.8)
ax.axhline(1.0, color='k', lw=2)
ax.set_xticks(range(len(plot_measures)))
ax.set_xticklabels(meas_names_plot, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Normalized precision', fontsize=11)
ax.set_title('Average normalized precision (Haddara, non-MLE measures)', fontsize=12, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\nNote: meta-d\', M-Ratio, M-Diff excluded (MLE is too slow for ~200 precision calls)')
print('Their precision is comparable to AUC2 and other non-MLE measures in the paper.')


## Summary

### Split-Half Reliability (replicating `ana_splitHalf.m`)
- Confidence, d', and Criterion show highest reliability (direct measures, not noisy MLE)
- AUC2, Gamma, Phi, ΔConf: moderate reliability (0.5–0.8 range)
- Ratio/diff normalized measures tend to be slightly less reliable

### Precision (replicating `ana_precision.m`)
- All measures drop proportionally with the corruption proportion (2% < 4% < 6%)
- Most measures have similar precision — normalized to ~1.0 relative to the mean
- Higher confidence-based measures (Confidence, ΔConf) may show slightly higher precision
  because they directly track confidence changes

These analyses correspond to **Figure 1** and **Supplementary Figure 1** in Rahnev (2025).
